# 3 · Create the AI/BI Dashboard

AI/BI Dashboards consume UC Business Semantics **metric views** natively. This
notebook takes the sample serialized dashboard
([./dashboard/dashboard.lvdash.json](./dashboard/dashboard.lvdash.json)), points it
at your catalog / schema, and **creates + publishes** it in your workspace.

Re-running is safe: if a dashboard with the same display name already exists it is
updated in place rather than duplicated.

### Install Dependencies

In [ ]:
!pip install databricks-sdk -U
dbutils.library.restartPython()

### Define Parameters

In [ ]:
dbutils.widgets.text("CATALOG_NAME", "workspace", "Catalog Name")
dbutils.widgets.text("SCHEMA_NAME", "retail_campaign", "Schema Name")
dbutils.widgets.text("WAREHOUSE_ID", "", "SQL Warehouse ID")
dbutils.widgets.text("DASHBOARD_NAME", "Retail Campaign Analytics - Sales Dashboard", "Dashboard Display Name")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
WAREHOUSE_ID = dbutils.widgets.get("WAREHOUSE_ID")
DASHBOARD_NAME = dbutils.widgets.get("DASHBOARD_NAME")

### Parameterize the Dashboard Definition

The sample `dashboard.lvdash.json` uses `CATALOG_NAME` / `SCHEMA_NAME` placeholders.
We substitute your parameter values into a copy the API can consume.

In [ ]:
with open('./dashboard/dashboard.lvdash.json', 'r') as f:
    serialized_dashboard = f.read()

serialized_dashboard = (
    serialized_dashboard
    .replace('CATALOG_NAME', CATALOG_NAME)
    .replace('SCHEMA_NAME', SCHEMA_NAME)
)
print('Dashboard definition parameterized for', f'{CATALOG_NAME}.{SCHEMA_NAME}')

### Create (or Update) and Publish the Dashboard

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard

w = WorkspaceClient()

# Idempotent: reuse an existing dashboard with the same display name
existing = next((d for d in w.lakeview.list() if d.display_name == DASHBOARD_NAME), None)

spec = Dashboard(
    display_name=DASHBOARD_NAME,
    serialized_dashboard=serialized_dashboard,
    warehouse_id=WAREHOUSE_ID,
)

if existing:
    dash = w.lakeview.update(dashboard_id=existing.dashboard_id, dashboard=spec)
    print('Updated existing dashboard:', dash.dashboard_id)
else:
    dash = w.lakeview.create(dashboard=spec)
    print('Created new dashboard:', dash.dashboard_id)

published = w.lakeview.publish(dash.dashboard_id, warehouse_id=WAREHOUSE_ID)
print('Published dashboard:', dash.dashboard_id)
print(f'{w.config.host}/dashboardsv3/{dash.dashboard_id}/published')

In [ ]:
dbutils.notebook.exit(dash.dashboard_id)